# 🪸 Servidor CoralSCOP + BioCLIP 2.5 ReefNet na Nuvem (Google Colab GPU)
### Pipeline Subaquático Completo: CoralSCOP ViT-B (CVPR 2024) + BioCLIP 2.5 ViT-H/14 ReefNet

Este notebook carrega simultaneamente **dois Foundation Models especializados em recifes de corais** na mesma GPU Tesla T4 (~3,4 GB de 15 GB de VRAM):

1. **Segmentação — CoralSCOP (`vit_b_coralscop.pth`, 382 MB):**
   - Treinado em **+41.000 fotos de recifes e +330.000 máscaras de corais** com ramo semântico exclusivo para colônias de corais (ignorando canos de PVC, etiquetas, fitas, fios de nylon, areia e rocha).
   - Inclui fechamento morfológico elíptico anti-fios (`7x7`) e contornos ordenados por área.
2. **Classificação Taxonômica de Gêneros — BioCLIP 2.5 ReefNet (`TsinghuaCorals/bioclip-2.5-vith14-reefnet-lora`):**
   - Foundation Model biológico (`ViT-H/14`, 214M imagens do *TreeOfLife-200M*) com ajuste fino no **ReefNet 1.0** (corais duros padronizados pelo WoRMS).
   - Classifica **exclusivamente em Gêneros de Corais** combinando recortes **Macro** (morfologia da colônia) + **Micro 224×224 em resolução nativa** (detalhe dos coralitos/septos), além de retornar assinaturas visuais de **1.024 dimensões** para suporte opcional a aprendizado *Few-Shot* em 1 clique!

In [ ]:
# 1. Verificar GPU alocada pelo Google Colab
!nvidia-smi

In [ ]:
# 2. Instalar dependências (CoralSCOP + OpenCLIP p/ BioCLIP 2.5) e aplicar patches p/ PyTorch 2.6+
import os

# Remover segment-anything genérico da Meta caso exista na sessão para não conflitar com o CoralSCOP
os.system("pip uninstall -y segment-anything > /dev/null 2>&1")
!pip install -q fastapi uvicorn pyngrok nest_asyncio python-multipart pillow opencv-python pycocotools huggingface_hub open_clip_torch safetensors

REPO_DIR = "/content/CoralSCOP"
if not os.path.exists(REPO_DIR):
    print("Clonando repositório oficial do CoralSCOP...")
    !git clone -q https://github.com/zhengziqiang/CoralSCOP.git {REPO_DIR}

# Patch 1: Compatibilidade com PyTorch 2.6+ (weights_only=False e map_location)
build_sam_path = os.path.join(REPO_DIR, "segment_anything", "build_sam.py")
with open(build_sam_path, "r", encoding="utf-8") as f:
    code_bs = f.read()
if "weights_only" not in code_bs:
    code_bs = code_bs.replace(
        "torch.load(f)",
        "torch.load(f, map_location='cpu', weights_only=False)"
    )
    with open(build_sam_path, "w", encoding="utf-8") as f:
        f.write(code_bs)

# Patch 2: Evitar erro dimensional quando um sub-quadrante de zoom não contém nenhum coral
amg_path = os.path.join(REPO_DIR, "segment_anything", "automatic_mask_generator.py")
with open(amg_path, "r", encoding="utf-8") as f:
    code_amg = f.read()
old_crop_line = 'data["crop_boxes"] = torch.tensor([crop_box for _ in range(len(data["rles"]))])'
new_crop_line = 'data["crop_boxes"] = torch.tensor([crop_box for _ in range(len(data["rles"]))], dtype=torch.long).reshape(-1, 4)'
if old_crop_line in code_amg:
    code_amg = code_amg.replace(old_crop_line, new_crop_line)
old_fc_block = 'fc_features=F.normalize(data["fc_features"],dim=-1)\n        similarity=fc_features.mm(fc_features.t())\n        similarity.fill_diagonal_(-np.inf)\n        data["similarity"]=similarity'
new_fc_block = (
    'if "fc_features" in data._stats and len(data["fc_features"]) > 0:\n'
    '            fc_features = F.normalize(data["fc_features"], dim=-1)\n'
    '            similarity = fc_features.mm(fc_features.t())\n'
    '            similarity.fill_diagonal_(-np.inf)\n'
    '            data["similarity"] = similarity\n'
    '        else:\n'
    '            data["similarity"] = torch.empty((0, 0))'
)
if old_fc_block in code_amg:
    code_amg = code_amg.replace(old_fc_block, new_fc_block)
with open(amg_path, "w", encoding="utf-8") as f:
    f.write(code_amg)

print("✅ Ambiente CoralSCOP + BioCLIP 2.5 preparado com sucesso!")

In [ ]:
# 3. Baixar os pesos oficiais do CoralSCOP (vit_b_coralscop.pth - 382 MB)
import os
import urllib.request
from huggingface_hub import hf_hub_download

os.makedirs("/content/CoralSCOP/checkpoints", exist_ok=True)
WEIGHTS_FILE = "/content/CoralSCOP/checkpoints/vit_b_coralscop.pth"

if not os.path.exists(WEIGHTS_FILE) or os.path.getsize(WEIGHTS_FILE) < 350_000_000:
    print("Baixando vit_b_coralscop.pth (382 MB) via Hugging Face CDN...")
    try:
        downloaded = hf_hub_download(
            repo_id="reefsupport/CoralSCOP",
            filename="models/vit_b_coralscop.pth",
            local_dir="/content/coralscop_hf"
        )
        if os.path.lexists(WEIGHTS_FILE):
            os.remove(WEIGHTS_FILE)
        os.symlink(downloaded, WEIGHTS_FILE)
        print("✅ Download do CoralSCOP concluído via Hugging Face!")
    except Exception as e:
        print(f"Aviso Hugging Face ({e}). Baixando do mirror oficial Dropbox...")
        DROPBOX_URL = "https://www.dropbox.com/scl/fi/pw5jiq9oc8e8kvkx1fdk0/vit_b_coralscop.pth?rlkey=qczdohnzxwgwoadpzeht0lim2&st=actcedwy&dl=1"
        urllib.request.urlretrieve(DROPBOX_URL, WEIGHTS_FILE)
        print("✅ Download concluído via mirror oficial!")
else:
    print("✅ Pesos do CoralSCOP (vit_b_coralscop.pth) já presentes no ambiente.")

In [ ]:
# 4. Carregar CoralSCOP ViT-B + BioCLIP 2.5 ViT-H/14 ReefNet na mesma GPU
import sys
import torch
import torch.nn.functional as F
import open_clip

# Garantir que o pacote segment_anything carregado seja o do CoralSCOP
for mod_name in list(sys.modules.keys()):
    if mod_name == "segment_anything" or mod_name.startswith("segment_anything."):
        del sys.modules[mod_name]
if "/content/CoralSCOP" not in sys.path:
    sys.path.insert(0, "/content/CoralSCOP")

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_label = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"[1/2] Carregando CoralSCOP ViT-B em: {device} ({gpu_label})...")

coralscop_model = sam_model_registry["vit_b"](checkpoint=WEIGHTS_FILE)
coralscop_model.to(device=device)
coralscop_model.eval()
print("✅ CoralSCOP ViT-B pronto!")

# Carregar BioCLIP 2.5 ViT-H/14 ReefNet (em fp16 na GPU para alta velocidade e baixo uso de VRAM)
BIOCLIP_CANDIDATES = [
    ("hf-hub:TsinghuaCorals/bioclip-2.5-vith14-reefnet-lora", "BioCLIP 2.5 ViT-H/14 ReefNet"),
    ("hf-hub:BobDerBaum/bioclip-2.5-vith14-reefnet-lora", "BioCLIP 2.5 ViT-H/14 ReefNet"),
    ("hf-hub:imageomics/bioclip-2.5-vith14", "BioCLIP 2.5 ViT-H/14"),
    ("hf-hub:imageomics/bioclip-2", "BioCLIP 2 ViT-L/14"),
]

bioclip_model = None
bioclip_preprocess = None
bioclip_tokenizer = None
BIOCLIP_LABEL = "Indisponível"

for repo_tag, label_name in BIOCLIP_CANDIDATES:
    try:
        print(f"[2/2] Baixando e carregando {label_name} ({repo_tag})...")
        prec = "fp16" if device == "cuda" else "fp32"
        bioclip_model, _, bioclip_preprocess = open_clip.create_model_and_transforms(
            repo_tag, device=device, precision=prec
        )
        bioclip_tokenizer = open_clip.get_tokenizer(repo_tag)
        bioclip_model.eval()
        BIOCLIP_LABEL = label_name
        break
    except Exception as err:
        print(f"Aviso ao tentar {repo_tag}: {err}")

if bioclip_model is None:
    raise RuntimeError("Falha ao carregar os modelos BioCLIP 2.5 / BioCLIP 2 do Hugging Face.")

# 11 Gêneros estritos de Corais do Atlântico Sul / Brasil (Zero classes abióticas ou não-corais)
CORAL_GENERA = [
    "Montastraea",
    "Mussismilia",
    "Porites",
    "Siderastrea",
    "Agaricia",
    "Millepora",
    "Madracis",
    "Meandrina",
    "Stephanocoenia",
    "Scolymia",
    "Favia"
]

GENUS_TAXONOMIC_PROMPTS = {
    "Montastraea": [
        "Animalia Cnidaria Anthozoa Scleractinia Montastraeidae Montastraea cavernosa",
        "Animalia Cnidaria Anthozoa Scleractinia Montastraeidae Montastraea",
        "underwater close-up of Montastraea cavernosa great star hard coral colony with large protruding plocoid corallites"
    ],
    "Mussismilia": [
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Mussismilia hispida",
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Mussismilia braziliensis",
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Mussismilia harttii",
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Mussismilia leptophylla",
        "Animalia Cnidaria Anthozoa Scleractinia Faviidae Mussismilia",
        "underwater close-up of Mussismilia massive Brazilian hard coral colony with large spinose toothed calices"
    ],
    "Porites": [
        "Animalia Cnidaria Anthozoa Scleractinia Poritidae Porites astreoides",
        "Animalia Cnidaria Anthozoa Scleractinia Poritidae Porites branneri",
        "Animalia Cnidaria Anthozoa Scleractinia Poritidae Porites",
        "underwater close-up of Porites astreoides mustard hill porous hard coral colony with tiny smooth pinhole corallites"
    ],
    "Siderastrea": [
        "Animalia Cnidaria Anthozoa Scleractinia Siderastreidae Siderastrea stellata",
        "Animalia Cnidaria Anthozoa Scleractinia Rhizangiidae Siderastrea stellata",
        "Animalia Cnidaria Anthozoa Scleractinia Siderastreidae Siderastrea siderea",
        "Animalia Cnidaria Anthozoa Scleractinia Siderastreidae Siderastrea radians",
        "underwater close-up of Siderastrea starlet encrusting hard coral colony with small cerioid star-shaped corallites"
    ],
    "Agaricia": [
        "Animalia Cnidaria Anthozoa Scleractinia Agariciidae Agaricia agaricites",
        "Animalia Cnidaria Anthozoa Scleractinia Agariciidae Agaricia humilis",
        "Animalia Cnidaria Anthozoa Scleractinia Agariciidae Agaricia fragilis",
        "underwater close-up of Agaricia lettuce plate hard coral colony with fine parallel ridges and valleys"
    ],
    "Millepora": [
        "Animalia Cnidaria Hydrozoa Anthoathecata Milleporidae Millepora alcicornis",
        "Animalia Cnidaria Hydrozoa Anthoathecata Milleporidae Millepora braziliensis",
        "Animalia Cnidaria Hydrozoa Anthoathecata Milleporidae Millepora nitida",
        "underwater close-up of Millepora smooth yellow-brown branching or blade fire hydrocoral colony with white tips"
    ],
    "Madracis": [
        "Animalia Cnidaria Anthozoa Scleractinia Astrocoeniidae Madracis decactis",
        "Animalia Cnidaria Anthozoa Scleractinia Pocilloporidae Madracis decactis",
        "Animalia Cnidaria Anthozoa Scleractinia Pocilloporidae Madracis",
        "underwater close-up of Madracis decactis nodular finger hard coral colony with ten-ray star septa"
    ],
    "Meandrina": [
        "Animalia Cnidaria Anthozoa Scleractinia Meandrinidae Meandrina braziliensis",
        "Animalia Cnidaria Anthozoa Scleractinia Meandrinidae Meandrina meandrites",
        "underwater close-up of Meandrina maze brain hard coral colony with sinuous valleys and thick bright septa"
    ],
    "Stephanocoenia": [
        "Animalia Cnidaria Anthozoa Scleractinia Astrocoeniidae Stephanocoenia intersepta",
        "Animalia Cnidaria Anthozoa Scleractinia Astrocoeniidae Stephanocoenia",
        "underwater close-up of Stephanocoenia intersepta blushing star hard coral colony with closely packed star corallites"
    ],
    "Scolymia": [
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Scolymia wellsi",
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Scolymia cubensis",
        "underwater close-up of Scolymia solitary circular fleshy disk hard coral polyp with radial septa"
    ],
    "Favia": [
        "Animalia Cnidaria Anthozoa Scleractinia Faviidae Favia gravida",
        "Animalia Cnidaria Anthozoa Scleractinia Mussidae Favia fragum",
        "Animalia Cnidaria Anthozoa Scleractinia Faviidae Favia",
        "underwater close-up of Favia gravida small golfball hard coral colony with raised oval corallites"
    ]
}

# Pré-computar embeddings taxonômicos L2-normalizados para os 11 Gêneros de Corais
GENUS_PROMPT_MATRICES = {}
GENUS_MEAN_VECTORS = {}

with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda"), dtype=torch.float16):
    for genus_name in CORAL_GENERA:
        tokens = bioclip_tokenizer(GENUS_TAXONOMIC_PROMPTS[genus_name]).to(device)
        t_feats = bioclip_model.encode_text(tokens).float()
        t_feats = F.normalize(t_feats, dim=-1)
        GENUS_PROMPT_MATRICES[genus_name] = t_feats
        GENUS_MEAN_VECTORS[genus_name] = F.normalize(t_feats.mean(dim=0), dim=-1)

vram_used = (torch.cuda.memory_allocated() / 1e9) if torch.cuda.is_available() else 0.0
print(f"🪸 CoralSCOP ViT-B + {BIOCLIP_LABEL} prontos na GPU! (VRAM total em uso: {vram_used:.2f} GB)")

In [ ]:
# 5. Criar a API FastAPI unificada (/segment para CoralSCOP e /classify_bioclip para BioCLIP 2.5)
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import io
import json
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from pycocotools import mask as mask_utils

app = FastAPI(title="CoralSCOP + BioCLIP 2.5 ReefNet Benthic Server")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

def score_embedding_against_coral_genera(vec_1d: torch.Tensor):
    """
    Calcula similaridade cosseno combinando o protótipo médio do gênero (55%)
    e a espécie/descrição mais próxima dentro do gênero (45%), retornando apenas gêneros de corais.
    """
    sims = {}
    for g in CORAL_GENERA:
        mean_s = float(torch.dot(vec_1d, GENUS_MEAN_VECTORS[g]).item())
        max_s = float(torch.mv(GENUS_PROMPT_MATRICES[g], vec_1d).max().item())
        sims[g] = round(0.55 * mean_s + 0.45 * max_s, 5)
    return sims

@app.get("/health")
def health():
    vram_gb = round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0
    return {
        "status": "online",
        "model": "CoralSCOP ViT-B (CVPR 2024)",
        "classifier_model": BIOCLIP_LABEL,
        "has_bioclip": True,
        "genera": CORAL_GENERA,
        "engine": "coralscop",
        "device": device,
        "vram_gb": vram_gb,
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
    }

@app.post("/segment")
async def segment_image(
    file: UploadFile = File(...),
    min_area: int = 100,
    pred_iou_thresh: float = 0.72,
    points_per_side: int = 32,
    stability_score_thresh: float = 0.62,
    crop_n_layers: int = 1,
    crop_n_points_downscale_factor: int = 2,
    bridge_strings: bool = True
):
    generator = SamAutomaticMaskGenerator(
        model=coralscop_model,
        points_per_side=int(points_per_side),
        pred_iou_thresh=float(pred_iou_thresh),
        stability_score_thresh=float(stability_score_thresh),
        crop_n_layers=int(crop_n_layers),
        crop_n_points_downscale_factor=int(crop_n_points_downscale_factor),
        min_mask_region_area=max(20, int(min_area))
    )

    contents = await file.read()
    image = Image.open(io.BytesIO(contents)).convert("RGB")
    image_np = np.array(image)
    h, w, _ = image_np.shape

    try:
        raw_masks = generator.generate(image_np)
    except (KeyError, IndexError) as empty_err:
        print(f"[Aviso] Nenhuma colônia detectada na imagem ({empty_err}).")
        raw_masks = []

    close_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

    masks_output = []
    for idx, m in enumerate(raw_masks, 1):
        seg_uint8 = (m["segmentation"].astype(np.uint8)) * 255
        if bridge_strings:
            seg_uint8 = cv2.morphologyEx(seg_uint8, cv2.MORPH_CLOSE, close_kernel)

        contours, _ = cv2.findContours(seg_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = sorted(contours, key=cv2.contourArea, reverse=True)

        simplified_polys = []
        for cnt in contours:
            if cv2.contourArea(cnt) < 20:
                continue
            approx = cv2.approxPolyDP(cnt, epsilon=1.4, closed=True)
            if len(approx) >= 3:
                simplified_polys.append(approx.reshape(-1, 2).tolist())

        if not simplified_polys:
            continue

        area = int(np.count_nonzero(seg_uint8))
        ys, xs = np.nonzero(seg_uint8)
        bx0, by0 = int(xs.min()), int(ys.min())
        bx1, by1 = int(xs.max()), int(ys.max())
        bbox = [bx0, by0, max(1, bx1 - bx0), max(1, by1 - by0)]

        M = cv2.moments(contours[0])
        if M["m00"] > 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
        else:
            cx = bx0 + bbox[2] // 2
            cy = by0 + bbox[3] // 2

        rle = mask_utils.encode(np.asfortranarray(seg_uint8 > 0))
        rle_dict = {"size": rle["size"], "counts": rle["counts"].decode("utf-8")}

        masks_output.append({
            "id": len(masks_output) + 1,
            "bbox": bbox,
            "centroid": [cy, cx],
            "area": area,
            "area_pct": (area / float(w * h)) * 100.0,
            "polygon": simplified_polys[0],
            "polygons": simplified_polys,
            "rle": rle_dict,
            "predicted_iou": float(m["predicted_iou"]),
            "stability_score": float(m["stability_score"])
        })

    return {
        "model": "CoralSCOP ViT-B (CVPR 2024)",
        "engine": "coralscop",
        "image_width": w,
        "image_height": h,
        "total_masks": len(masks_output),
        "masks": masks_output
    }

@app.post("/classify_bioclip")
async def classify_bioclip(
    file: UploadFile = File(...),
    metadata_json: str = Form(...)
):
    meta = json.loads(metadata_json)
    tile_size = int(meta.get("tile_size", 224))
    cols = int(meta.get("cols", 10))
    total_tiles = int(meta.get("total_tiles", 0))
    colonies_meta = meta.get("colonies", [])

    if total_tiles == 0 or not colonies_meta:
        return {"classifier_model": BIOCLIP_LABEL, "results": []}

    contents = await file.read()
    sheet = Image.open(io.BytesIO(contents)).convert("RGB")

    preprocessed_tensors = []
    for idx in range(total_tiles):
        r = idx // cols
        c = idx % cols
        tile = sheet.crop((c * tile_size, r * tile_size, (c + 1) * tile_size, (r + 1) * tile_size))
        preprocessed_tensors.append(bioclip_preprocess(tile))

    visual_dtype = next(bioclip_model.visual.parameters()).dtype
    feat_chunks = []
    batch_size = 64

    with torch.no_grad(), torch.amp.autocast("cuda", enabled=(device == "cuda"), dtype=torch.float16):
        for start in range(0, len(preprocessed_tensors), batch_size):
            batch = torch.stack(preprocessed_tensors[start:start + batch_size]).to(
                device=device, dtype=visual_dtype
            )
            feats = bioclip_model.encode_image(batch).float()
            feats = F.normalize(feats, dim=-1)
            feat_chunks.append(feats)

    all_feats = torch.cat(feat_chunks, dim=0)

    results = []
    for col_info in colonies_meta:
        macro_idx = col_info.get("macro_indices", [])
        micro_idx = col_info.get("micro_indices", [])
        all_idx = macro_idx + micro_idx

        f_macro = F.normalize(all_feats[macro_idx].mean(dim=0), dim=-1) if macro_idx else all_feats[all_idx[0]]
        f_micro = F.normalize(all_feats[micro_idx].mean(dim=0), dim=-1) if micro_idx else f_macro

        # Fusão Multi-Escala: 35% morfologia Macro da colônia + 65% coralitos Micro 224x224 (padrão ReefNet)
        fused = F.normalize(0.35 * f_macro + 0.65 * f_micro, dim=-1)
        zero_shot_sims = score_embedding_against_coral_genera(fused)
        top_genus = max(zero_shot_sims, key=zero_shot_sims.get)

        # Consenso entre as múltiplas vistas (macro + micro-pontos)
        agree = 0
        for v_idx in all_idx:
            v_sims = score_embedding_against_coral_genera(all_feats[v_idx])
            if max(v_sims, key=v_sims.get) == top_genus:
                agree += 1
        consensus = float(agree / max(1, len(all_idx)))

        results.append({
            "id": col_info["id"],
            "embedding": [round(float(x), 5) for x in fused.cpu().tolist()],
            "zero_shot_sims": zero_shot_sims,
            "consensus_ratio": round(consensus, 3),
            "views_count": len(all_idx)
        })

    return {
        "classifier_model": BIOCLIP_LABEL,
        "genera": CORAL_GENERA,
        "results": results
    }


In [ ]:
# @title 6. Iniciar Túnel Seguro (ngrok ou Localtunnel) e Servidor FastAPI
# @markdown Insira seu **NGROK_AUTHTOKEN** abaixo (ou salve nos Secrets do Colab com a chave `NGROK_AUTHTOKEN`).
# @markdown *Se deixar em branco, o notebook usará o **Localtunnel** automaticamente como fallback.*

NGROK_AUTHTOKEN = "" # @param {type:"string"}

import nest_asyncio
import uvicorn
import sys
import os
import subprocess
import time
import asyncio

token = NGROK_AUTHTOKEN.strip()
if not token:
    try:
        from google.colab import userdata
        token = (userdata.get('NGROK_AUTHTOKEN') or '').strip()
    except Exception as sec_err:
        err_name = type(sec_err).__name__
        if 'NotebookAccess' in err_name:
            print('⚠️ ATENÇÃO: Seu NGROK_AUTHTOKEN está salvo nos Secrets do Colab, mas o acesso para este novo notebook está DESATIVADO!')
            print('👉 Clique no ícone de Chave 🔑 (Secrets) na barra esquerda do Colab, ative a chave ao lado de NGROK_AUTHTOKEN e rode esta Célula 6 novamente!')
        token = os.environ.get("NGROK_AUTHTOKEN", "").strip()

public_url = None

if token:
    print("🔒 NGROK_AUTHTOKEN detectado! Iniciando túnel oficial do ngrok (Alta Estabilidade)...")
    try:
        from pyngrok import ngrok
        ngrok.set_auth_token(token)
        ngrok.kill()
        tunnel = ngrok.connect(8000, "http")
        public_url = tunnel.public_url
        if public_url.startswith("http://"):
            public_url = public_url.replace("http://", "https://")

        print("\n" + "=" * 72)
        print("🪸 SERVIDOR CORALSCOP + BIOCLIP 2.5 REEFNET ON-LINE COM NGROK!")
        print(f"URL: {public_url}")
        print("Copie a URL acima e cole na interface web local!")
        print("=" * 72 + "\n")
    except Exception as e:
        print(f"❌ Falha ao iniciar ngrok: {e}")
        print("Alternando automaticamente para Localtunnel...")
        token = ""

if not token or not public_url:
    print("🌐 Nenhum NGROK_AUTHTOKEN ativo. Usando Localtunnel...")
    os.system("npm install -g localtunnel > /dev/null 2>&1")
    process = subprocess.Popen(["lt", "--port", "8000"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    time.sleep(5)
    for _ in range(5):
        if process.poll() is not None:
            break
        line = process.stdout.readline().strip()
        if "your url is:" in line.lower():
            public_url = line.split("your url is:")[-1].strip()
            break
        time.sleep(1)

    try:
        ip = subprocess.check_output(["curl", "ipv4.icanhazip.com"], text=True).strip()
    except Exception:
        ip = "IP Indisponível"

    print("\n" + "=" * 72)
    print("🪸 SERVIDOR CORALSCOP + BIOCLIP 2.5 REEFNET ON-LINE COM LOCALTUNNEL!")
    print(f"URL: {public_url}")
    print(f"Se pedir Tunnel Password, use o IP: {ip}")
    print("=" * 72 + "\n")

nest_asyncio.apply()
if "_coralscop_uvicorn_server" in globals() and _coralscop_uvicorn_server is not None:
    _coralscop_uvicorn_server.should_exit = True
    time.sleep(0.5)

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
_coralscop_uvicorn_server = uvicorn.Server(config)

try:
    loop = asyncio.get_event_loop()
    loop.create_task(_coralscop_uvicorn_server.serve())
    print("🚀 Servidor FastAPI (CoralSCOP + BioCLIP 2.5) rodando no Colab com sucesso!")
except Exception as e:
    print(f"Falha ao acoplar o servidor ao loop ativo: {e}")
